# LEDs TUTO 

In [1]:
from pynq import Overlay, MMIO
import time

file = "./leds.bit"
ol = Overlay(file)
print (" Overlay chargé")

 Overlay chargé


In [2]:
ol.ip_dict

{'button': {'type': 'xilinx.com:ip:axi_gpio:2.0',
  'mem_id': 'S_AXI',
  'memtype': 'REGISTER',
  'gpio': {},
  'interrupts': {},
  'parameters': {'C_ALL_INPUTS': '1',
   'C_ALL_INPUTS_2': '0',
   'C_ALL_OUTPUTS': '0',
   'C_ALL_OUTPUTS_2': '0',
   'C_DOUT_DEFAULT': '0x00000000',
   'C_DOUT_DEFAULT_2': '0x00000000',
   'C_FAMILY': 'zynq',
   'C_GPIO2_WIDTH': '32',
   'C_GPIO_WIDTH': '4',
   'C_INTERRUPT_PRESENT': '0',
   'C_IS_DUAL': '0',
   'C_S_AXI_ADDR_WIDTH': '9',
   'C_S_AXI_DATA_WIDTH': '32',
   'C_TRI_DEFAULT': '0xFFFFFFFF',
   'C_TRI_DEFAULT_2': '0xFFFFFFFF',
   'Component_Name': 'leds_axi_gpio_0_1',
   'GPIO2_BOARD_INTERFACE': 'Custom',
   'GPIO_BOARD_INTERFACE': 'btns_4bits',
   'USE_BOARD_FLOW': 'true',
   'EDK_IPTYPE': 'PERIPHERAL',
   'C_BASEADDR': '0x41200000',
   'C_HIGHADDR': '0x4120FFFF',
   'ADDR_WIDTH': '9',
   'ARUSER_WIDTH': '0',
   'AWUSER_WIDTH': '0',
   'BUSER_WIDTH': '0',
   'CLK_DOMAIN': 'leds_processing_system7_0_0_FCLK_CLK0',
   'DATA_WIDTH': '32',
   'FREQ_

# Recuperation des adresses

In [3]:
leds_addr = ol.ip_dict["leds"]["phys_addr"]
sw_addr = ol.ip_dict["switches"]["phys_addr"]
btn_addr = ol.ip_dict["button"]["phys_addr"]
rgb_addr = ol.ip_dict["rgb_leds"]["phys_addr"]

# Création des MMIO 

In [19]:
leds = MMIO(leds_addr, 0x0008)
sw = MMIO(sw_addr, 0x10)
btn = MMIO(btn_addr, 0x0008)
rgb = MMIO(rgb_addr, 0x010)

# Registre Offset 

In [4]:
GPIO_DATA = 0x0 # registre de données
GPIO_TRI = 0x4 # registre de direction

# Configuration des E/S 

In [20]:
leds.write(GPIO_TRI, 0x0)
rgb.write(GPIO_TRI, 0x0)
sw.write(GPIO_TRI, 0xFFFFFFFF)
btn.write(GPIO_TRI, 0xFFFFFFFF)

# Different TEST


In [24]:
valeur = sw.read(GPIO_DATA)
print(valeur)

2


In [28]:
valeur = btn.read(GPIO_DATA)
print(valeur)

5


In [29]:
leds.write(GPIO_DATA, 0b1010) # corresponds au 4 leds, de droite a gauche, 1er led a droite = 0001, ect. La on va allumer la 4 et 2 eme leds
rgb.write(GPIO_DATA,0b100100) # 2 leds rgb chacun sur 3 bits, 100 = rouge, 010 = vert, 001 = bleu, ect...

In [30]:
leds.write(GPIO_DATA, 0b0000) 
rgb.write(GPIO_DATA,0b000000)

# Code principale 

In [36]:
#def de different mode
def chenillard():
    leds.write(GPIO_DATA, 0b0001)
    time.sleep(0.2)
    ch_value = sw.read(GPIO_DATA) & 0b11
    if ch_value != 0b01: # si les switchs sont pas dans cette config ( s1 off et s0 on) on stop 
        return
    
    leds.write(GPIO_DATA, 0b0010)
    time.sleep(0.2)
    if (sw.read(GPIO_DATA) & 0b11) != 0b01: 
        return

    leds.write(GPIO_DATA, 0b0100)
    time.sleep(0.2)
    if (sw.read(GPIO_DATA) & 0b11) != 0b01: 
        return

    leds.write(GPIO_DATA, 0b1000)
    time.sleep(0.2)
    if (sw.read(GPIO_DATA) & 0b11) != 0b01: 
        return
    
    leds.write(GPIO_DATA, 0b0000)


def saut():
    leds.write(GPIO_DATA, 0b0101)
    time.sleep(0.3)
    
    saut_value = sw.read(GPIO_DATA) & 0b11
    if saut_value != 0b10:
        return
    leds.write(GPIO_DATA, 0b1010)
    time.sleep(0.3)

In [43]:
counter = 0

red = 0b100100
vert = 0b001001
bleu = 0b010010

off_rgb = 0b000000
off_led = 0b0000

pause = False

while True:
    # lecture des SW
    sw_value = sw.read(GPIO_DATA) & 0b11
    s0 = sw_value & 0b01
    s1 = (sw_value >> 1) & 0b01
    
    #Lecture des BTN
    btn_value = btn.read(GPIO_DATA) & 0b1111
    b0 = btn_value & 0b0001
    b1 = (btn_value >> 1) & 0b0001
    b2 = (btn_value >> 2) & 0b0001
    
    # Fonctions des BTN
    if b0 == 1: # BTN STOP
        print(" Fin du programme", end = "\r")
        rgb.write(GPIO_DATA, off_rgb)
        leds.write(GPIO_DATA, off_led)
        break
    if b1 == 1:
        pause = True
        print (" Pause du programme...  ", end = "\r")
        time.sleep(0.2)
    if b2 == 1:
        pause = False
        print (" Reprise du programme...", end = "\r")
        time.sleep(0.2)
    if pause:
        time.sleep(0.2)
        continue
    
    # 00 / Rien
    if s1 == 0 and s0 == 0:
        rgb.write(GPIO_DATA, off_rgb)
        leds.write(GPIO_DATA, off_led)
        time.sleep(0.05)
    
    #01 / Chennilard
    elif s1 == 0 and s0 == 1:
        rgb.write(GPIO_DATA, red)
        chenillard()
    
    #10 / saut
    elif s1 == 1 and s0 == 0:
        rgb.write(GPIO_DATA, vert)
        saut()
        
    #11 / compteur
    elif s1 == 1 and s0 == 1:
        rgb.write(GPIO_DATA, bleu)
        leds.write(GPIO_DATA, counter)
        time.sleep(0.4)
        counter = (counter + 1) % 16

        
        
        
        
        
